# 提取全局特征（GTA)

In [ ]:
# 提取局部结构特征第一步
import os

structure_path = './data/structures/GTA/'
folders = os.listdir(structure_path)

f = open('./Protein_extract_feature_step1.sh', 'w')
for folder in folders:
    directory = os.path.join(structure_path, folder)
    files = os.listdir(directory)
    for file in files:
        f.write(f"python ./Extract_feature_step1.py {file} {folder} 6 6 GTA\n")
f.close()

# nohup parallel --jobs 30 < Protein_extract_feature_step1.sh > Feature_Step_1.out 2>&1 &


# 提取局部特征（GTA）

In [2]:
# 提取局部结构特征第二步
import os

structure_path = './data/structures/GTA/'
folders = os.listdir(structure_path)

sample_redies_udp = 6
sample_redies_sugar = 6

step = 1
epoch = 1
f = open(f'./Extract_feature_step2_epoch{epoch}.sh', 'w')
for folder in folders:
    directory = os.path.join(structure_path, folder)
    files = os.listdir(directory)
    for file in files:
        if step % 40 == 0:
            f.write(f"taskset -c {step} python ./Extract_feature_step2.py {file} {folder} {sample_redies_udp} {sample_redies_sugar} GTA\n")
            f.close()
            epoch += 1
            f = open(f'./Extract_feature_step2_epoch{epoch}.sh', 'w')
            step = 1
            continue
        f.write(f"taskset -c {step} python ./Extract_feature_step2.py {file} {folder} {sample_redies_udp} {sample_redies_sugar} GTA\n")
        step = step + 1
f.close()

f = open(f'./Extract_feature_step2_AAArun.sh', 'w')
for e in range(epoch):
    f.write(f"parallel --jobs 40 < Extract_feature_step2_epoch{e+1}.sh\n")
f.close()

# nohup bash Extract_feature_step2_AAArun.sh &


# 生成数据集 - 排除NaN的数据（GTA）

In [1]:
skip_list = [
    'GT2_AUV51559_1', 'GT2_QQN43717_1',                    # 网格边数小于100
    'GT2_CAO99139_1',                                      # 不能前向传播，含NaN，fold1，test
    'GT2_QXH51152_1', 'GT2_UXJ04752_1',                    # 不能前向传播，含NaN，fold1，validation
    'GT2_BBN74818_1', 'GT2_XBN37388_1', 'GT2_AUZ44912_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_QIY47078_1', 'GT2_BBX21645_1', 'GT2_BCZ64624_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_UQX61489_1', 'GT2_ATY61741_1', 'GT2_UAY64771_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_UXV20753_1', 'GT2_QQJ12945_1', 'GT2_BBN74818_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_AIT50810_1'                                       # 不能前向传播，含NaN，fold1，train
]

skip_correspond_information_dict = { # 文件中的数字 ：报错的时候找不到的数字
    '1.25, 22.89, 14.67, 0.30, 0.00, -0.02, 0.62###-0.25, 15.41, 3.61, 0.62, 0.51, -0.02, -1.00###-1.08, 13.70, 9.15, -0.79, -0.02, -0.33, -0.78\n':
    '1.25, 22.89, 14.67, 0.30, 0.00, -0.02, 0.62###-0.25, 15.41, 3.61, 0.62, 0.51, -0.02, -1.00###-1.08, 13.70, 9.15, -0.79, -0.01, -0.33, -0.78\n',   # 不能索引名称，数字错误，fold1， train
    '14.18, 13.08, 6.65, 0.60, 0.00, -0.38, -0.78###15.18, 16.51, 0.84, 0.83, 0.00, -0.25, -0.18###-2.30, 22.16, 14.71, 0.69, 0.00, -0.03, -0.09\n':
    '14.18, 13.08, 6.65, 0.60, 0.00, -0.38, -0.78###15.18, 16.51, 0.84, 0.83, 0.00, -0.25, -0.18###-2.30, 22.16, 14.71, 0.68, 0.00, -0.03, -0.09\n',   # 不能索引名称，数字错误，fold1， train
    '11.23, 8.35, 2.35, -0.88, -0.01, 0.01, -0.16###4.17, 13.97, 8.67, 0.62, -0.56, -1.00, -0.78###7.83, 14.11, 14.14, 0.78, 0.00, -0.30, 0.84\n':
    '11.23, 8.35, 2.35, -0.88, -0.01, 0.01, -0.15###4.17, 13.97, 8.67, 0.62, -0.56, -1.00, -0.78###7.83, 14.11, 14.14, 0.78, 0.00, -0.30, 0.84\n',     # 不能索引名称，数字错误，fold1， train
    '0.91, 16.63, 15.25, 0.29, 0.38, 0.26, 0.84###11.99, 8.96, 2.82, -0.82, 0.00, -0.37, -0.29###16.39, 11.44, 4.52, 0.32, 0.00, -0.08, -0.29\n':
    '0.91, 16.63, 15.25, 0.29, 0.38, 0.26, 0.84###11.99, 8.96, 2.82, -0.82, 0.00, -0.37, -0.29###16.39, 11.44, 4.52, 0.31, 0.00, -0.08, -0.29\n'       # 不能索引名称，数字错误，fold1， train
}

skip_correspond_information = list(skip_correspond_information_dict.keys())

In [2]:
'''
根据DiffPool的教程，我需要准备的东西有4样，分别是A、graph_indicator、graph_labels、node_attributes
'''
import numpy as np
import os
from tqdm import tqdm
import shutil
import pandas as pd

sample_redies_udp = 6
sample_redies_sugar = 6

graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                    'UDP-Gal': 3, 'UDP-GalNAc': 4,
                    'UDP-Xyl': 5, 'GDP-Man': 6,
                    'dTDP-Rha': 7, 'Other': 8}

for fold_num in range(1, 11):
    # 获取有哪些local结构
    storage_path = './data/local_features/GTA/'
    dl_data_path = './data/dl_data/GTA/'
    dl_data_path = os.path.join(dl_data_path, f"fold{fold_num}")

    folders = os.listdir(storage_path)
    local_dict = {}
    for folder in folders:
        npy_path = os.path.join(storage_path, folder)
        local_dict[folder.split('_')[0]] = [x.split('.npy')[0] for x in os.listdir(npy_path)]

    # 获取活性标签
    df = pd.read_excel('./data/GTA_training_data.xlsx')
    activate_dict = {}
    for i in range(0,df.shape[0]):
        if '[-]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Bifunction'
        elif '[*]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Other'
        else:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['NDP-Sugar活性'][i]

    # 生成文件列表
    process_path = {'train':[], 'validation':[], 'test':[], 'nova':[]}
    df = pd.read_excel(f'./data/cluster/GTA/dataseat_split_{fold_num}.xlsx')
    for i in range(0,df.shape[0]):
        if df['GenBank'][i] in local_dict[df['Family'][i]]:
            if df['GenBank'][i] in skip_list:
                continue
            npy_path = os.path.join(storage_path, f"{df['Family'][i]}_aln_check")
            process_path[df['Dataset'][i]].append(os.path.join(npy_path, df['GenBank'][i]+'.npy'))
    train_process_path = process_path['train']
    validation_process_path = process_path['validation']
    test_process_path = process_path['test']
    nova_process_path = process_path['nova']

    # 管理文件夹
    if not os.path.isdir(dl_data_path):
        os.makedirs(dl_data_path, exist_ok=True)
    else:
        shutil.rmtree(dl_data_path)
        os.makedirs(dl_data_path, exist_ok=True)
    os.makedirs(f'{dl_data_path}/train/')
    os.makedirs(f'{dl_data_path}/train/trace_file/')
    os.makedirs(f'{dl_data_path}/validation/')
    os.makedirs(f'{dl_data_path}/validation/trace_file/')
    os.makedirs(f'{dl_data_path}/test/')
    os.makedirs(f'{dl_data_path}/test/trace_file/')
    os.makedirs(f'{dl_data_path}/nova/')
    os.makedirs(f'{dl_data_path}/nova/trace_file/')

    # ==================== 生成数据 ====================
    def make_database(process_path: list, data_type: str):
        f_A = open(f'{dl_data_path}/{data_type}/GTmining_A.txt', 'w')
        f_graph_indicator = open(f'{dl_data_path}/{data_type}/GTmining_graph_indicator.txt', 'w')
        f_graph_labels = open(f'{dl_data_path}/{data_type}/GTmining_graph_labels.txt', 'w')
        f_node_attributes = open(f'{dl_data_path}/{data_type}/GTmining_node_attributes.txt', 'w')
        f_correspond_information = open(f'{dl_data_path}/{data_type}/Predict_correspond_information.txt', 'w')
        edge_max = -1
        graph_idx = -1
        # 已经重构了数据处理部分，读取一次npy文件，写入四种不同的文件，提升I/O性能
        for file in tqdm(process_path):
            # +++++ 一次性读取npy文件 +++++
            try:
                input_dict = np.load(file, allow_pickle=True)
            except:
                print(f"wrong {file}")
                continue
            input_dict = input_dict[()]
            if len(input_dict['edges']) <= 100:
                # 用来检查不正确的局部网格
                print(f"error local feature in {file.split('/')[-1].split('.npy')[0]}")
                continue
            # +++++ f_A的数据处理 +++++
            edges = input_dict['edges']
            edges += (edge_max +1)
            for edge in edges:
                f_A.write(f"{edge[0]}, {edge[1]}\n")
            edge_max = np.max(edges)
            # +++++ f_graph_indicator的数据处理 +++++
            graph_idx += 1
            xyzs = input_dict['xyz']
            for i in range(0, xyzs.shape[0]):
                f_graph_indicator.write(f"{graph_idx}\n")
            # +++++ f_graph_labels的数据处理 +++++
            group_name = file.split('/')[-1].split('.npy')[0]
            f_graph_labels.write(f"{graph_label_dict[activate_dict[group_name]]}\n")
            # +++++ f_node_attributes的数据处理 +++++
            xyzs = input_dict['xyz']
            sis = input_dict['si']
            hbonds = input_dict['hbond']
            charges = input_dict['charge']
            hphobs = input_dict['hphob']
            trace_temp = file.split('/')[-1] + f'==={edge_max}.txt'
            f_trace_file = open(f'{dl_data_path}/{data_type}/trace_file/{trace_temp}', 'w')
            for i in range(0, xyzs.shape[0]):
                # x = xyzs[i][0] + np.random.normal(loc=0, scale=0.75)
                # y = xyzs[i][1] + np.random.normal(loc=0, scale=0.75)
                # z = xyzs[i][2] + np.random.normal(loc=0, scale=0.75)
                x = xyzs[i][0]
                y = xyzs[i][1]
                z = xyzs[i][2]
                f_trace_file.write("{:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z))
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], charges[i][0], hphobs[i][0]))
            f_trace_file.close()
            f_correspond_information.write(file.split('/')[-1].split('.npy')[0] + '===')
            temp_correspond_information = ''
            temp_correspond_information = temp_correspond_information + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[0][0], 5),
                                                                                                                round(xyzs[0][1], 5),
                                                                                                                round(xyzs[0][2], 5),
                                                                                                                round(sis[0][0], 5),
                                                                                                                round(hbonds[0][0], 5),
                                                                                                                round(charges[0][0], 5),
                                                                                                                round(hphobs[0][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[1][0], 5),
                                                                                                                    round(xyzs[1][1], 5),
                                                                                                                    round(xyzs[1][2], 5),
                                                                                                                    round(sis[1][0], 5),
                                                                                                                    round(hbonds[1][0], 5),
                                                                                                                    round(charges[1][0], 5),
                                                                                                                    round(hphobs[1][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}\n".format(round(xyzs[2][0], 5),
                                                                                                                    round(xyzs[2][1], 5),
                                                                                                                    round(xyzs[2][2], 5),
                                                                                                                    round(sis[2][0], 5),
                                                                                                                    round(hbonds[2][0], 5),
                                                                                                                    round(charges[2][0], 5),
                                                                                                                    round(hphobs[2][0], 5))
            if temp_correspond_information in skip_correspond_information:
                temp_correspond_information = skip_correspond_information_dict[temp_correspond_information]
            f_correspond_information.write(temp_correspond_information)

        f_A.close()
        f_graph_indicator.close()
        f_graph_labels.close()
        f_node_attributes.close()
        f_correspond_information.close()

    make_database(train_process_path, 'train')
    make_database(validation_process_path, 'validation')
    make_database(test_process_path, 'test')
    make_database(nova_process_path, 'nova')

100%|██████████| 475/475 [00:02<00:00, 189.33it/s]


然后去DiffPool里面运行测试脚本，测试是否有错误，以及能否和trace文件对应起来
- calculate_features_SI.ipynb

# 生成数据集 - 排除NaN的数据（GTA）- 使用全部数据

In [1]:
skip_list = [
    'GT2_AUV51559_1', 'GT2_QQN43717_1',                    # 网格边数小于100
    'GT2_CAO99139_1',                                      # 不能前向传播，含NaN，fold1，test
    'GT2_QXH51152_1', 'GT2_UXJ04752_1',                    # 不能前向传播，含NaN，fold1，validation
    'GT2_BBN74818_1', 'GT2_XBN37388_1', 'GT2_AUZ44912_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_QIY47078_1', 'GT2_BBX21645_1', 'GT2_BCZ64624_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_UQX61489_1', 'GT2_ATY61741_1', 'GT2_UAY64771_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_UXV20753_1', 'GT2_QQJ12945_1', 'GT2_BBN74818_1',  # 不能前向传播，含NaN，fold1，train
    'GT2_AIT50810_1'                                       # 不能前向传播，含NaN，fold1，train
]

skip_correspond_information_dict = { # 文件中的数字 ：报错的时候找不到的数字
    '1.25, 22.89, 14.67, 0.30, 0.00, -0.02, 0.62###-0.25, 15.41, 3.61, 0.62, 0.51, -0.02, -1.00###-1.08, 13.70, 9.15, -0.79, -0.02, -0.33, -0.78\n':
    '1.25, 22.89, 14.67, 0.30, 0.00, -0.02, 0.62###-0.25, 15.41, 3.61, 0.62, 0.51, -0.02, -1.00###-1.08, 13.70, 9.15, -0.79, -0.01, -0.33, -0.78\n',   # 不能索引名称，数字错误，fold1， train
    '14.18, 13.08, 6.65, 0.60, 0.00, -0.38, -0.78###15.18, 16.51, 0.84, 0.83, 0.00, -0.25, -0.18###-2.30, 22.16, 14.71, 0.69, 0.00, -0.03, -0.09\n':
    '14.18, 13.08, 6.65, 0.60, 0.00, -0.38, -0.78###15.18, 16.51, 0.84, 0.83, 0.00, -0.25, -0.18###-2.30, 22.16, 14.71, 0.68, 0.00, -0.03, -0.09\n',   # 不能索引名称，数字错误，fold1， train
    '11.23, 8.35, 2.35, -0.88, -0.01, 0.01, -0.16###4.17, 13.97, 8.67, 0.62, -0.56, -1.00, -0.78###7.83, 14.11, 14.14, 0.78, 0.00, -0.30, 0.84\n':
    '11.23, 8.35, 2.35, -0.88, -0.01, 0.01, -0.15###4.17, 13.97, 8.67, 0.62, -0.56, -1.00, -0.78###7.83, 14.11, 14.14, 0.78, 0.00, -0.30, 0.84\n',     # 不能索引名称，数字错误，fold1， train
    '0.91, 16.63, 15.25, 0.29, 0.38, 0.26, 0.84###11.99, 8.96, 2.82, -0.82, 0.00, -0.37, -0.29###16.39, 11.44, 4.52, 0.32, 0.00, -0.08, -0.29\n':
    '0.91, 16.63, 15.25, 0.29, 0.38, 0.26, 0.84###11.99, 8.96, 2.82, -0.82, 0.00, -0.37, -0.29###16.39, 11.44, 4.52, 0.31, 0.00, -0.08, -0.29\n'       # 不能索引名称，数字错误，fold1， train
}

skip_correspond_information = list(skip_correspond_information_dict.keys())

In [2]:
'''
根据DiffPool的教程，我需要准备的东西有4样，分别是A、graph_indicator、graph_labels、node_attributes
'''
import numpy as np
import os
from tqdm import tqdm
import shutil
import pandas as pd

sample_redies_udp = 6
sample_redies_sugar = 6

graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                    'UDP-Gal': 3, 'UDP-GalNAc': 4,
                    'UDP-Xyl': 5, 'GDP-Man': 6,
                    'dTDP-Rha': 7, 'Other': 8}

for fold_num in range(1, 11):
    # 获取有哪些local结构
    storage_path = './data/local_features/GTA/'
    dl_data_path = './data/dl_data/GTA_alldata/'
    dl_data_path = os.path.join(dl_data_path, f"fold{fold_num}")

    folders = os.listdir(storage_path)
    local_dict = {}
    for folder in folders:
        npy_path = os.path.join(storage_path, folder)
        local_dict[folder.split('_')[0]] = [x.split('.npy')[0] for x in os.listdir(npy_path)]

    # 获取活性标签
    df = pd.read_excel('./data/GTA_training_data.xlsx')
    activate_dict = {}
    for i in range(0,df.shape[0]):
        if '[-]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Bifunction'
        elif '[*]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Other'
        else:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['NDP-Sugar活性'][i]

    # 生成文件列表
    process_path = {'train':[], 'validation':[], 'test':[]}
    df = pd.read_excel(f'./data/cluster/GTA_alldata/dataseat_split_{fold_num}.xlsx')
    for i in range(0,df.shape[0]):
        if df['GenBank'][i] in local_dict[df['Family'][i]]:
            if df['GenBank'][i] in skip_list:
                continue
            npy_path = os.path.join(storage_path, f"{df['Family'][i]}_aln_check")
            process_path[df['Dataset'][i]].append(os.path.join(npy_path, df['GenBank'][i]+'.npy'))
    train_process_path = process_path['train']
    validation_process_path = process_path['validation']
    test_process_path = process_path['test']

    # 管理文件夹
    if not os.path.isdir(dl_data_path):
        os.makedirs(dl_data_path, exist_ok=True)
    else:
        shutil.rmtree(dl_data_path)
        os.makedirs(dl_data_path, exist_ok=True)
    os.makedirs(f'{dl_data_path}/train/')
    os.makedirs(f'{dl_data_path}/train/trace_file/')
    os.makedirs(f'{dl_data_path}/validation/')
    os.makedirs(f'{dl_data_path}/validation/trace_file/')
    os.makedirs(f'{dl_data_path}/test/')
    os.makedirs(f'{dl_data_path}/test/trace_file/')

    # ==================== 生成数据 ====================
    def make_database(process_path: list, data_type: str):
        f_A = open(f'{dl_data_path}/{data_type}/GTmining_A.txt', 'w')
        f_graph_indicator = open(f'{dl_data_path}/{data_type}/GTmining_graph_indicator.txt', 'w')
        f_graph_labels = open(f'{dl_data_path}/{data_type}/GTmining_graph_labels.txt', 'w')
        f_node_attributes = open(f'{dl_data_path}/{data_type}/GTmining_node_attributes.txt', 'w')
        f_correspond_information = open(f'{dl_data_path}/{data_type}/Predict_correspond_information.txt', 'w')
        edge_max = -1
        graph_idx = -1
        # 已经重构了数据处理部分，读取一次npy文件，写入四种不同的文件，提升I/O性能
        for file in tqdm(process_path):
            # +++++ 一次性读取npy文件 +++++
            try:
                input_dict = np.load(file, allow_pickle=True)
            except:
                print(f"wrong {file}")
                continue
            input_dict = input_dict[()]
            if len(input_dict['edges']) <= 100:
                # 用来检查不正确的局部网格
                print(f"error local feature in {file.split('/')[-1].split('.npy')[0]}")
                continue
            # +++++ f_A的数据处理 +++++
            edges = input_dict['edges']
            edges += (edge_max +1)
            for edge in edges:
                f_A.write(f"{edge[0]}, {edge[1]}\n")
            edge_max = np.max(edges)
            # +++++ f_graph_indicator的数据处理 +++++
            graph_idx += 1
            xyzs = input_dict['xyz']
            for i in range(0, xyzs.shape[0]):
                f_graph_indicator.write(f"{graph_idx}\n")
            # +++++ f_graph_labels的数据处理 +++++
            group_name = file.split('/')[-1].split('.npy')[0]
            f_graph_labels.write(f"{graph_label_dict[activate_dict[group_name]]}\n")
            # +++++ f_node_attributes的数据处理 +++++
            xyzs = input_dict['xyz']
            sis = input_dict['si']
            hbonds = input_dict['hbond']
            charges = input_dict['charge']
            hphobs = input_dict['hphob']
            trace_temp = file.split('/')[-1] + f'==={edge_max}.txt'
            f_trace_file = open(f'{dl_data_path}/{data_type}/trace_file/{trace_temp}', 'w')
            for i in range(0, xyzs.shape[0]):
                # x = xyzs[i][0] + np.random.normal(loc=0, scale=0.75)
                # y = xyzs[i][1] + np.random.normal(loc=0, scale=0.75)
                # z = xyzs[i][2] + np.random.normal(loc=0, scale=0.75)
                x = xyzs[i][0]
                y = xyzs[i][1]
                z = xyzs[i][2]
                f_trace_file.write("{:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z))
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], charges[i][0], hphobs[i][0]))
            f_trace_file.close()
            f_correspond_information.write(file.split('/')[-1].split('.npy')[0] + '===')
            temp_correspond_information = ''
            temp_correspond_information = temp_correspond_information + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[0][0], 5),
                                                                                                                round(xyzs[0][1], 5),
                                                                                                                round(xyzs[0][2], 5),
                                                                                                                round(sis[0][0], 5),
                                                                                                                round(hbonds[0][0], 5),
                                                                                                                round(charges[0][0], 5),
                                                                                                                round(hphobs[0][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[1][0], 5),
                                                                                                                    round(xyzs[1][1], 5),
                                                                                                                    round(xyzs[1][2], 5),
                                                                                                                    round(sis[1][0], 5),
                                                                                                                    round(hbonds[1][0], 5),
                                                                                                                    round(charges[1][0], 5),
                                                                                                                    round(hphobs[1][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}\n".format(round(xyzs[2][0], 5),
                                                                                                                    round(xyzs[2][1], 5),
                                                                                                                    round(xyzs[2][2], 5),
                                                                                                                    round(sis[2][0], 5),
                                                                                                                    round(hbonds[2][0], 5),
                                                                                                                    round(charges[2][0], 5),
                                                                                                                    round(hphobs[2][0], 5))
            if temp_correspond_information in skip_correspond_information:
                temp_correspond_information = skip_correspond_information_dict[temp_correspond_information]
            f_correspond_information.write(temp_correspond_information)

        f_A.close()
        f_graph_indicator.close()
        f_graph_labels.close()
        f_node_attributes.close()
        f_correspond_information.close()

    make_database(train_process_path, 'train')
    make_database(validation_process_path, 'validation')
    make_database(test_process_path, 'test')

100%|██████████| 1305/1305 [00:07<00:00, 184.29it/s]


然后去DiffPool里面运行测试脚本，测试是否有错误，以及能否和trace文件对应起来
- calculate_features_SI.ipynb

# 提取全局特征（GTB)

In [24]:
# 提取局部结构特征第一步
import os

structure_path = './data/structures/GTB/'
folders = os.listdir(structure_path)

f = open('./Protein_extract_feature_step1.sh', 'w')
for folder in folders:
    directory = os.path.join(structure_path, folder)
    files = os.listdir(directory)
    for file in files:
        f.write(f"python ./Extract_feature_step1.py {file} {folder} 6 6 GTB\n")
f.close()

# nohup parallel --jobs 30 < Protein_extract_feature_step1.sh > Feature_Step_1.out 2>&1 &


# 提取局部特征（GTB）

In [1]:
# 提取局部结构特征第二步
import os

structure_path = './data/structures/GTB/'
folders = os.listdir(structure_path)

sample_redies_udp = 6
sample_redies_sugar = 6

step = 1
epoch = 1
f = open(f'./Extract_feature_step2_epoch{epoch}.sh', 'w')
for folder in folders:
    directory = os.path.join(structure_path, folder)
    files = os.listdir(directory)
    for file in files:
        if step % 40 == 0:
            f.write(f"taskset -c {step} python ./Extract_feature_step2.py {file} {folder} {sample_redies_udp} {sample_redies_sugar} GTB\n")
            f.close()
            epoch += 1
            f = open(f'./Extract_feature_step2_epoch{epoch}.sh', 'w')
            step = 1
            continue
        f.write(f"taskset -c {step} python ./Extract_feature_step2.py {file} {folder} {sample_redies_udp} {sample_redies_sugar} GTB\n")
        step = step + 1
f.close()

f = open(f'./Extract_feature_step2_AAArun.sh', 'w')
for e in range(epoch):
    f.write(f"parallel --jobs 40 < Extract_feature_step2_epoch{e+1}.sh\n")
f.close()

# nohup bash Extract_feature_step2_AAArun.sh &


# 生成数据集 - 排除NaN的数据（GTB）

In [1]:
skip_list = [
    'GT35_AHY09627_1', 'GT19_BBB60788_1',                         # 不能前向传播，含NaN，fold1，test
    'GT10_AAW32622_1', 'GT106_CAB4104418_1', 'GT35_QKX50649_1',   # 不能前向传播，含NaN，fold1，validation
    'GT19_WYA60860_1', 'GT56_UXY10792_1', 'GT35_QIX92616_1',      # 不能前向传播，含NaN，fold1，validation
    'GT30_QNF15040_1', 'GT35_WWU74798_1',                         # 不能前向传播，含NaN，fold1，validation
    'GT70_APO94310_1', 'GT70_QTK46120_1', 'GT5_AGB47314_1',       # 不能前向传播，含NaN，fold1，train
    'GT19_AZA48355_1', 'GT104_UXB11094_1', 'GT30_QOV30031_1',     # 不能前向传播，含NaN，fold1，train
    'GT4_UDD40513_1', 'GT4_AYU93913_1', 'GT56_QHA87116_1',        # 不能前向传播，含NaN，fold1，train
    'GT4_BCO86114_1', 'GT30_UTV85551_1', 'GT5_CAI9087173_1',      # 不能前向传播，含NaN，fold1，train
    'GT20_QLV68278_1', 'GT4_UHD26605_1', 'GT4_BBA87692_1',        # 不能前向传播，含NaN，fold1，train
    'GT4_QGS26923_1', 'GT19_ABQ29269_1', 'GT30_QQC33163_1',       # 不能前向传播，含NaN，fold1，train
    'GT30_UWQ44439_1', 'GT4_CAI34053_2', 'GT4_ALI28300_1',        # 不能前向传播，含NaN，fold1，train
    'GT5_WQB98038_1', 'GT4_CCC19979_1', 'GT30_ARG39747_1',        # 不能前向传播，含NaN，fold1，train
    'GT19_XAU06904_1', 'GT35_ACM59680_1', 'GT56_ALB53317_1',      # 不能前向传播，含NaN，fold1，train
    'GT113_VFH88248_1'                                            # 不能前向传播，含NaN，fold1，train
    # 不能前向传播，含NaN，fold1，train
    # 不能前向传播，含NaN，fold1，train
]

skip_correspond_information_dict = { # 文件中的数字 ：报错的时候找不到的数字
    '5.98, -7.13, -3.93, -0.43, 0.86, 0.49, 0.42###-3.07, 2.25, -1.01, 0.84, 0.00, 0.35, 0.62###-2.41, -6.33, -9.28, 0.38, 0.00, 0.31, 0.42\n':
    '5.98, -7.13, -3.93, -0.43, 0.87, 0.49, 0.42###-3.07, 2.25, -1.01, 0.84, 0.00, 0.35, 0.62###-2.41, -6.33, -9.28, 0.38, 0.00, 0.31, 0.42\n',               # 不能索引名称，数字错误，fold1， train
    '7.42, -13.24, -7.28, -0.88, 0.00, -0.08, 0.93###6.70, -14.21, -1.82, -0.61, -0.98, -0.14, -0.18###9.80, -11.00, -0.21, -0.94, 0.00, -0.12, -0.18\n':
    '7.42, -13.24, -7.28, -0.88, 0.00, -0.08, 0.93###6.70, -14.21, -1.82, -0.62, -0.98, -0.14, -0.18###9.80, -11.00, -0.21, -0.94, 0.00, -0.12, -0.18\n',     # 不能索引名称，数字错误，fold1， train
    '0.82, -8.32, -9.76, 0.77, 0.00, 0.05, 0.93###-3.04, -12.05, -2.13, 0.88, 0.00, 0.07, 0.93###-8.65, -1.38, 2.92, 0.55, -0.01, -0.48, -0.78\n':
    '0.82, -8.32, -9.76, 0.77, 0.00, 0.06, 0.93###-3.04, -12.05, -2.13, 0.88, 0.00, 0.07, 0.93###-8.65, -1.38, 2.92, 0.55, -0.01, -0.48, -0.78\n',            # 不能索引名称，数字错误，fold1， train
    '1.68, -5.78, -1.32, -0.79, 0.00, 0.29, -0.97###0.53, -11.05, -1.25, 0.58, 0.00, 0.09, 0.62###5.65, -14.01, -7.41, 0.65, 0.00, -0.02, -0.78\n':
    '1.68, -5.78, -1.32, -0.79, 0.00, 0.29, -0.97###0.53, -11.05, -1.25, 0.58, 0.00, 0.09, 0.62###5.65, -14.01, -7.41, 0.64, 0.00, -0.02, -0.78\n',           # 不能索引名称，数字错误，fold1， train
    '-6.24, 3.77, 3.26, -0.61, 0.00, 0.58, -0.87###7.78, -9.30, -5.01, 0.33, -0.00, -0.07, -0.09###2.66, -0.76, 0.61, -0.46, 0.00, 0.22, -0.78\n':
    '-6.24, 3.77, 3.26, -0.61, 0.00, 0.58, -0.87###7.78, -9.30, -5.01, 0.34, -0.00, -0.07, -0.09###2.66, -0.76, 0.61, -0.46, 0.00, 0.22, -0.78\n',            # 不能索引名称，数字错误，fold1， train
    '0.73, -4.93, -9.51, 0.28, 0.00, 0.64, -0.71###-4.69, -12.66, -6.07, -0.76, 0.27, 0.69, -0.17###-2.83, -7.21, -4.56, 0.78, -0.08, 0.55, -0.29\n':
    '0.73, -4.93, -9.51, 0.28, 0.00, 0.64, -0.71###-4.69, -12.66, -6.07, -0.76, 0.27, 0.69, -0.17###-2.83, -7.21, -4.56, 0.78, -0.09, 0.55, -0.29\n',         # 不能索引名称，数字错误，fold1， train
    '-1.14, 2.80, -2.53, 0.36, -0.79, 0.36, -0.16###-1.36, 4.61, -0.09, -0.82, 0.02, 1.00, -0.33###5.40, -12.38, -8.49, 0.55, 0.00, 0.44, 0.84\n':
    '-1.14, 2.80, -2.53, 0.36, -0.79, 0.36, -0.16###-1.36, 4.61, -0.09, -0.82, 0.01, 1.00, -0.33###5.40, -12.38, -8.49, 0.55, 0.00, 0.44, 0.84\n',            # 不能索引名称，数字错误，fold1， train
    '-2.98, -7.75, 3.41, -1.00, 0.00, -0.10, 1.00###3.96, 1.99, -7.74, -0.14, 0.00, 0.05, 0.41###1.29, -8.34, -10.72, -0.72, -0.04, 0.15, 0.93\n':
    '-2.98, -7.75, 3.41, -1.00, 0.00, -0.11, 1.00###3.96, 1.99, -7.74, -0.14, 0.00, 0.05, 0.41###1.29, -8.34, -10.72, -0.72, -0.04, 0.15, 0.93\n',            # 不能索引名称，数字错误，fold1， train
    '-0.82, -1.17, 1.77, -0.55, 0.00, 0.42, 0.84###-5.49, 0.46, 3.64, -0.74, 0.00, 0.37, 0.93###-2.84, 0.56, -3.21, -0.17, 0.00, 0.56, -0.87\n':
    '-0.82, -1.17, 1.77, -0.55, 0.00, 0.42, 0.84###-5.49, 0.46, 3.64, -0.74, 0.00, 0.37, 0.93###-2.84, 0.56, -3.21, -0.16, 0.00, 0.56, -0.87\n'               # 不能索引名称，数字错误，fold1， train
}

skip_correspond_information = list(skip_correspond_information_dict.keys())

In [2]:
'''
根据DiffPool的教程，我需要准备的东西有4样，分别是A、graph_indicator、graph_labels、node_attributes
'''
import numpy as np
import os
from tqdm import tqdm
import shutil
import pandas as pd

sample_redies_udp = 6
sample_redies_sugar = 6

graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                    'UDP-Gal': 3, 'UDP-GalNAc': 4,
                    'UDP-Xyl': 5, 'GDP-Man': 6, 'GDP-Fuc': 7,
                    'dTDP-Rha': 8, 'Other': 9}

for fold_num in range(1, 11):
    # 获取有哪些local结构
    storage_path = './data/local_features/GTB/'
    dl_data_path = './data/dl_data/GTB/'
    dl_data_path = os.path.join(dl_data_path, f"fold{fold_num}")

    folders = os.listdir(storage_path)
    local_dict = {}
    for folder in folders:
        npy_path = os.path.join(storage_path, folder)
        local_dict[folder.split('_')[0]] = [x.split('.npy')[0] for x in os.listdir(npy_path)]

    # 获取活性标签
    df = pd.read_excel('./data/GTB_training_data.xlsx')
    activate_dict = {}
    for i in range(0,df.shape[0]):
        if '[-]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Bifunction'
        elif '[*]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Other'
        else:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['NDP-Sugar活性'][i]

    # 生成文件列表
    process_path = {'train':[], 'validation':[], 'test':[], 'nova':[]}
    df = pd.read_excel(f'./data/cluster/GTB/dataseat_split_{fold_num}.xlsx')
    for i in range(0,df.shape[0]):
        if df['GenBank'][i] in local_dict[df['Family'][i]]:
            if df['GenBank'][i] in skip_list:
                continue
            npy_path = os.path.join(storage_path, f"{df['Family'][i]}_aln_check")
            process_path[df['Dataset'][i]].append(os.path.join(npy_path, df['GenBank'][i]+'.npy'))
    train_process_path = process_path['train']
    validation_process_path = process_path['validation']
    test_process_path = process_path['test']
    nova_process_path = process_path['nova']

    # 管理文件夹
    if not os.path.isdir(dl_data_path):
        os.makedirs(dl_data_path, exist_ok=True)
    else:
        shutil.rmtree(dl_data_path)
        os.makedirs(dl_data_path, exist_ok=True)
    os.makedirs(f'{dl_data_path}/train/')
    os.makedirs(f'{dl_data_path}/train/trace_file/')
    os.makedirs(f'{dl_data_path}/validation/')
    os.makedirs(f'{dl_data_path}/validation/trace_file/')
    os.makedirs(f'{dl_data_path}/test/')
    os.makedirs(f'{dl_data_path}/test/trace_file/')
    os.makedirs(f'{dl_data_path}/nova/')
    os.makedirs(f'{dl_data_path}/nova/trace_file/')

    # ==================== 生成数据 ====================
    def make_database(process_path: list, data_type: str):
        f_A = open(f'{dl_data_path}/{data_type}/GTmining_A.txt', 'w')
        f_graph_indicator = open(f'{dl_data_path}/{data_type}/GTmining_graph_indicator.txt', 'w')
        f_graph_labels = open(f'{dl_data_path}/{data_type}/GTmining_graph_labels.txt', 'w')
        f_node_attributes = open(f'{dl_data_path}/{data_type}/GTmining_node_attributes.txt', 'w')
        f_correspond_information = open(f'{dl_data_path}/{data_type}/Predict_correspond_information.txt', 'w')
        edge_max = -1
        graph_idx = -1
        # 已经重构了数据处理部分，读取一次npy文件，写入四种不同的文件，提升I/O性能
        for file in tqdm(process_path):
            # +++++ 一次性读取npy文件 +++++
            try:
                input_dict = np.load(file, allow_pickle=True)
            except:
                print(f"wrong {file}")
                continue
            input_dict = input_dict[()]
            if len(input_dict['edges']) <= 100:
                # 用来检查不正确的局部网格
                print(f"error local feature in {file.split('/')[-1].split('.npy')[0]}")
                continue
            # +++++ f_A的数据处理 +++++
            edges = input_dict['edges']
            edges += (edge_max +1)
            for edge in edges:
                f_A.write(f"{edge[0]}, {edge[1]}\n")
            edge_max = np.max(edges)
            # +++++ f_graph_indicator的数据处理 +++++
            graph_idx += 1
            xyzs = input_dict['xyz']
            for i in range(0, xyzs.shape[0]):
                f_graph_indicator.write(f"{graph_idx}\n")
            # +++++ f_graph_labels的数据处理 +++++
            group_name = file.split('/')[-1].split('.npy')[0]
            f_graph_labels.write(f"{graph_label_dict[activate_dict[group_name]]}\n")
            # +++++ f_node_attributes的数据处理 +++++
            xyzs = input_dict['xyz']
            sis = input_dict['si']
            hbonds = input_dict['hbond']
            charges = input_dict['charge']
            hphobs = input_dict['hphob']
            trace_temp = file.split('/')[-1] + f'==={edge_max}.txt'
            f_trace_file = open(f'{dl_data_path}/{data_type}/trace_file/{trace_temp}', 'w')
            for i in range(0, xyzs.shape[0]):
                # x = xyzs[i][0] + np.random.normal(loc=0, scale=0.75)
                # y = xyzs[i][1] + np.random.normal(loc=0, scale=0.75)
                # z = xyzs[i][2] + np.random.normal(loc=0, scale=0.75)
                x = xyzs[i][0]
                y = xyzs[i][1]
                z = xyzs[i][2]
                f_trace_file.write("{:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z))
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], charges[i][0], hphobs[i][0]))
            f_trace_file.close()
            f_correspond_information.write(file.split('/')[-1].split('.npy')[0] + '===')
            temp_correspond_information = ''
            temp_correspond_information = temp_correspond_information + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[0][0], 5),
                                                                                                                round(xyzs[0][1], 5),
                                                                                                                round(xyzs[0][2], 5),
                                                                                                                round(sis[0][0], 5),
                                                                                                                round(hbonds[0][0], 5),
                                                                                                                round(charges[0][0], 5),
                                                                                                                round(hphobs[0][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[1][0], 5),
                                                                                                                    round(xyzs[1][1], 5),
                                                                                                                    round(xyzs[1][2], 5),
                                                                                                                    round(sis[1][0], 5),
                                                                                                                    round(hbonds[1][0], 5),
                                                                                                                    round(charges[1][0], 5),
                                                                                                                    round(hphobs[1][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}\n".format(round(xyzs[2][0], 5),
                                                                                                                    round(xyzs[2][1], 5),
                                                                                                                    round(xyzs[2][2], 5),
                                                                                                                    round(sis[2][0], 5),
                                                                                                                    round(hbonds[2][0], 5),
                                                                                                                    round(charges[2][0], 5),
                                                                                                                    round(hphobs[2][0], 5))
            if temp_correspond_information in skip_correspond_information:
                temp_correspond_information = skip_correspond_information_dict[temp_correspond_information]
            f_correspond_information.write(temp_correspond_information)

        f_A.close()
        f_graph_indicator.close()
        f_graph_labels.close()
        f_node_attributes.close()
        f_correspond_information.close()

    make_database(train_process_path, 'train')
    make_database(validation_process_path, 'validation')
    make_database(test_process_path, 'test')
    make_database(nova_process_path, 'nova')

100%|██████████| 2194/2194 [00:13<00:00, 159.80it/s]


然后去DiffPool里面运行测试脚本，测试是否有错误，以及能否和trace文件对应起来
- calculate_features_SI.ipynb

# 生成数据集 - 排除NaN的数据（GTB）- 使用全部数据

In [1]:
skip_list = [
    'GT35_AHY09627_1', 'GT19_BBB60788_1',                         # 不能前向传播，含NaN，fold1，test
    'GT10_AAW32622_1', 'GT106_CAB4104418_1', 'GT35_QKX50649_1',   # 不能前向传播，含NaN，fold1，validation
    'GT19_WYA60860_1', 'GT56_UXY10792_1', 'GT35_QIX92616_1',      # 不能前向传播，含NaN，fold1，validation
    'GT30_QNF15040_1', 'GT35_WWU74798_1',                         # 不能前向传播，含NaN，fold1，validation
    'GT70_APO94310_1', 'GT70_QTK46120_1', 'GT5_AGB47314_1',       # 不能前向传播，含NaN，fold1，train
    'GT19_AZA48355_1', 'GT104_UXB11094_1', 'GT30_QOV30031_1',     # 不能前向传播，含NaN，fold1，train
    'GT4_UDD40513_1', 'GT4_AYU93913_1', 'GT56_QHA87116_1',        # 不能前向传播，含NaN，fold1，train
    'GT4_BCO86114_1', 'GT30_UTV85551_1', 'GT5_CAI9087173_1',      # 不能前向传播，含NaN，fold1，train
    'GT20_QLV68278_1', 'GT4_UHD26605_1', 'GT4_BBA87692_1',        # 不能前向传播，含NaN，fold1，train
    'GT4_QGS26923_1', 'GT19_ABQ29269_1', 'GT30_QQC33163_1',       # 不能前向传播，含NaN，fold1，train
    'GT30_UWQ44439_1', 'GT4_CAI34053_2', 'GT4_ALI28300_1',        # 不能前向传播，含NaN，fold1，train
    'GT5_WQB98038_1', 'GT4_CCC19979_1', 'GT30_ARG39747_1',        # 不能前向传播，含NaN，fold1，train
    'GT19_XAU06904_1', 'GT35_ACM59680_1', 'GT56_ALB53317_1',      # 不能前向传播，含NaN，fold1，train
    'GT113_VFH88248_1'                                            # 不能前向传播，含NaN，fold1，train
    # 不能前向传播，含NaN，fold1，train
    # 不能前向传播，含NaN，fold1，train
]

skip_correspond_information_dict = { # 文件中的数字 ：报错的时候找不到的数字
    '5.98, -7.13, -3.93, -0.43, 0.86, 0.49, 0.42###-3.07, 2.25, -1.01, 0.84, 0.00, 0.35, 0.62###-2.41, -6.33, -9.28, 0.38, 0.00, 0.31, 0.42\n':
    '5.98, -7.13, -3.93, -0.43, 0.87, 0.49, 0.42###-3.07, 2.25, -1.01, 0.84, 0.00, 0.35, 0.62###-2.41, -6.33, -9.28, 0.38, 0.00, 0.31, 0.42\n',               # 不能索引名称，数字错误，fold1， train
    '7.42, -13.24, -7.28, -0.88, 0.00, -0.08, 0.93###6.70, -14.21, -1.82, -0.61, -0.98, -0.14, -0.18###9.80, -11.00, -0.21, -0.94, 0.00, -0.12, -0.18\n':
    '7.42, -13.24, -7.28, -0.88, 0.00, -0.08, 0.93###6.70, -14.21, -1.82, -0.62, -0.98, -0.14, -0.18###9.80, -11.00, -0.21, -0.94, 0.00, -0.12, -0.18\n',     # 不能索引名称，数字错误，fold1， train
    '0.82, -8.32, -9.76, 0.77, 0.00, 0.05, 0.93###-3.04, -12.05, -2.13, 0.88, 0.00, 0.07, 0.93###-8.65, -1.38, 2.92, 0.55, -0.01, -0.48, -0.78\n':
    '0.82, -8.32, -9.76, 0.77, 0.00, 0.06, 0.93###-3.04, -12.05, -2.13, 0.88, 0.00, 0.07, 0.93###-8.65, -1.38, 2.92, 0.55, -0.01, -0.48, -0.78\n',            # 不能索引名称，数字错误，fold1， train
    '1.68, -5.78, -1.32, -0.79, 0.00, 0.29, -0.97###0.53, -11.05, -1.25, 0.58, 0.00, 0.09, 0.62###5.65, -14.01, -7.41, 0.65, 0.00, -0.02, -0.78\n':
    '1.68, -5.78, -1.32, -0.79, 0.00, 0.29, -0.97###0.53, -11.05, -1.25, 0.58, 0.00, 0.09, 0.62###5.65, -14.01, -7.41, 0.64, 0.00, -0.02, -0.78\n',           # 不能索引名称，数字错误，fold1， train
    '-6.24, 3.77, 3.26, -0.61, 0.00, 0.58, -0.87###7.78, -9.30, -5.01, 0.33, -0.00, -0.07, -0.09###2.66, -0.76, 0.61, -0.46, 0.00, 0.22, -0.78\n':
    '-6.24, 3.77, 3.26, -0.61, 0.00, 0.58, -0.87###7.78, -9.30, -5.01, 0.34, -0.00, -0.07, -0.09###2.66, -0.76, 0.61, -0.46, 0.00, 0.22, -0.78\n',            # 不能索引名称，数字错误，fold1， train
    '0.73, -4.93, -9.51, 0.28, 0.00, 0.64, -0.71###-4.69, -12.66, -6.07, -0.76, 0.27, 0.69, -0.17###-2.83, -7.21, -4.56, 0.78, -0.08, 0.55, -0.29\n':
    '0.73, -4.93, -9.51, 0.28, 0.00, 0.64, -0.71###-4.69, -12.66, -6.07, -0.76, 0.27, 0.69, -0.17###-2.83, -7.21, -4.56, 0.78, -0.09, 0.55, -0.29\n',         # 不能索引名称，数字错误，fold1， train
    '-1.14, 2.80, -2.53, 0.36, -0.79, 0.36, -0.16###-1.36, 4.61, -0.09, -0.82, 0.02, 1.00, -0.33###5.40, -12.38, -8.49, 0.55, 0.00, 0.44, 0.84\n':
    '-1.14, 2.80, -2.53, 0.36, -0.79, 0.36, -0.16###-1.36, 4.61, -0.09, -0.82, 0.01, 1.00, -0.33###5.40, -12.38, -8.49, 0.55, 0.00, 0.44, 0.84\n',            # 不能索引名称，数字错误，fold1， train
    '-2.98, -7.75, 3.41, -1.00, 0.00, -0.10, 1.00###3.96, 1.99, -7.74, -0.14, 0.00, 0.05, 0.41###1.29, -8.34, -10.72, -0.72, -0.04, 0.15, 0.93\n':
    '-2.98, -7.75, 3.41, -1.00, 0.00, -0.11, 1.00###3.96, 1.99, -7.74, -0.14, 0.00, 0.05, 0.41###1.29, -8.34, -10.72, -0.72, -0.04, 0.15, 0.93\n',            # 不能索引名称，数字错误，fold1， train
    '-0.82, -1.17, 1.77, -0.55, 0.00, 0.42, 0.84###-5.49, 0.46, 3.64, -0.74, 0.00, 0.37, 0.93###-2.84, 0.56, -3.21, -0.17, 0.00, 0.56, -0.87\n':
    '-0.82, -1.17, 1.77, -0.55, 0.00, 0.42, 0.84###-5.49, 0.46, 3.64, -0.74, 0.00, 0.37, 0.93###-2.84, 0.56, -3.21, -0.16, 0.00, 0.56, -0.87\n'               # 不能索引名称，数字错误，fold1， train
}

skip_correspond_information = list(skip_correspond_information_dict.keys())

In [2]:
'''
根据DiffPool的教程，我需要准备的东西有4样，分别是A、graph_indicator、graph_labels、node_attributes
'''
import numpy as np
import os
from tqdm import tqdm
import shutil
import pandas as pd

sample_redies_udp = 6
sample_redies_sugar = 6

graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                    'UDP-Gal': 3, 'UDP-GalNAc': 4,
                    'UDP-Xyl': 5, 'GDP-Man': 6, 'GDP-Fuc': 7,
                    'dTDP-Rha': 8, 'Other': 9}

for fold_num in range(1, 11):
    # 获取有哪些local结构
    storage_path = './data/local_features/GTB/'
    dl_data_path = './data/dl_data/GTB_alldata/'
    dl_data_path = os.path.join(dl_data_path, f"fold{fold_num}")

    folders = os.listdir(storage_path)
    local_dict = {}
    for folder in folders:
        npy_path = os.path.join(storage_path, folder)
        local_dict[folder.split('_')[0]] = [x.split('.npy')[0] for x in os.listdir(npy_path)]

    # 获取活性标签
    df = pd.read_excel('./data/GTB_training_data.xlsx')
    activate_dict = {}
    for i in range(0,df.shape[0]):
        if '[-]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Bifunction'
        elif '[*]' in df['NDP-Sugar活性'][i]:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Other'
        else:
            activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['NDP-Sugar活性'][i]

    # 生成文件列表
    process_path = {'train':[], 'validation':[], 'test':[]}
    df = pd.read_excel(f'./data/cluster/GTB_alldata/dataseat_split_{fold_num}.xlsx')
    for i in range(0,df.shape[0]):
        if df['GenBank'][i] in local_dict[df['Family'][i]]:
            if df['GenBank'][i] in skip_list:
                continue
            npy_path = os.path.join(storage_path, f"{df['Family'][i]}_aln_check")
            process_path[df['Dataset'][i]].append(os.path.join(npy_path, df['GenBank'][i]+'.npy'))
    train_process_path = process_path['train']
    validation_process_path = process_path['validation']
    test_process_path = process_path['test']


    # 管理文件夹
    if not os.path.isdir(dl_data_path):
        os.makedirs(dl_data_path, exist_ok=True)
    else:
        shutil.rmtree(dl_data_path)
        os.makedirs(dl_data_path, exist_ok=True)
    os.makedirs(f'{dl_data_path}/train/')
    os.makedirs(f'{dl_data_path}/train/trace_file/')
    os.makedirs(f'{dl_data_path}/validation/')
    os.makedirs(f'{dl_data_path}/validation/trace_file/')
    os.makedirs(f'{dl_data_path}/test/')
    os.makedirs(f'{dl_data_path}/test/trace_file/')

    # ==================== 生成数据 ====================
    def make_database(process_path: list, data_type: str):
        f_A = open(f'{dl_data_path}/{data_type}/GTmining_A.txt', 'w')
        f_graph_indicator = open(f'{dl_data_path}/{data_type}/GTmining_graph_indicator.txt', 'w')
        f_graph_labels = open(f'{dl_data_path}/{data_type}/GTmining_graph_labels.txt', 'w')
        f_node_attributes = open(f'{dl_data_path}/{data_type}/GTmining_node_attributes.txt', 'w')
        f_correspond_information = open(f'{dl_data_path}/{data_type}/Predict_correspond_information.txt', 'w')
        edge_max = -1
        graph_idx = -1
        # 已经重构了数据处理部分，读取一次npy文件，写入四种不同的文件，提升I/O性能
        for file in tqdm(process_path):
            # +++++ 一次性读取npy文件 +++++
            try:
                input_dict = np.load(file, allow_pickle=True)
            except:
                print(f"wrong {file}")
                continue
            input_dict = input_dict[()]
            if len(input_dict['edges']) <= 100:
                # 用来检查不正确的局部网格
                print(f"error local feature in {file.split('/')[-1].split('.npy')[0]}")
                continue
            # +++++ f_A的数据处理 +++++
            edges = input_dict['edges']
            edges += (edge_max +1)
            for edge in edges:
                f_A.write(f"{edge[0]}, {edge[1]}\n")
            edge_max = np.max(edges)
            # +++++ f_graph_indicator的数据处理 +++++
            graph_idx += 1
            xyzs = input_dict['xyz']
            for i in range(0, xyzs.shape[0]):
                f_graph_indicator.write(f"{graph_idx}\n")
            # +++++ f_graph_labels的数据处理 +++++
            group_name = file.split('/')[-1].split('.npy')[0]
            f_graph_labels.write(f"{graph_label_dict[activate_dict[group_name]]}\n")
            # +++++ f_node_attributes的数据处理 +++++
            xyzs = input_dict['xyz']
            sis = input_dict['si']
            hbonds = input_dict['hbond']
            charges = input_dict['charge']
            hphobs = input_dict['hphob']
            trace_temp = file.split('/')[-1] + f'==={edge_max}.txt'
            f_trace_file = open(f'{dl_data_path}/{data_type}/trace_file/{trace_temp}', 'w')
            for i in range(0, xyzs.shape[0]):
                # x = xyzs[i][0] + np.random.normal(loc=0, scale=0.75)
                # y = xyzs[i][1] + np.random.normal(loc=0, scale=0.75)
                # z = xyzs[i][2] + np.random.normal(loc=0, scale=0.75)
                x = xyzs[i][0]
                y = xyzs[i][1]
                z = xyzs[i][2]
                f_trace_file.write("{:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z))
                f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], charges[i][0], hphobs[i][0]))
            f_trace_file.close()
            f_correspond_information.write(file.split('/')[-1].split('.npy')[0] + '===')
            temp_correspond_information = ''
            temp_correspond_information = temp_correspond_information + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[0][0], 5),
                                                                                                                round(xyzs[0][1], 5),
                                                                                                                round(xyzs[0][2], 5),
                                                                                                                round(sis[0][0], 5),
                                                                                                                round(hbonds[0][0], 5),
                                                                                                                round(charges[0][0], 5),
                                                                                                                round(hphobs[0][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[1][0], 5),
                                                                                                                    round(xyzs[1][1], 5),
                                                                                                                    round(xyzs[1][2], 5),
                                                                                                                    round(sis[1][0], 5),
                                                                                                                    round(hbonds[1][0], 5),
                                                                                                                    round(charges[1][0], 5),
                                                                                                                    round(hphobs[1][0], 5))
            temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}\n".format(round(xyzs[2][0], 5),
                                                                                                                    round(xyzs[2][1], 5),
                                                                                                                    round(xyzs[2][2], 5),
                                                                                                                    round(sis[2][0], 5),
                                                                                                                    round(hbonds[2][0], 5),
                                                                                                                    round(charges[2][0], 5),
                                                                                                                    round(hphobs[2][0], 5))
            if temp_correspond_information in skip_correspond_information:
                temp_correspond_information = skip_correspond_information_dict[temp_correspond_information]
            f_correspond_information.write(temp_correspond_information)

        f_A.close()
        f_graph_indicator.close()
        f_graph_labels.close()
        f_node_attributes.close()
        f_correspond_information.close()

    make_database(train_process_path, 'train')
    make_database(validation_process_path, 'validation')
    make_database(test_process_path, 'test')


100%|██████████| 2109/2109 [00:27<00:00, 75.60it/s] 


然后去DiffPool里面运行测试脚本，测试是否有错误，以及能否和trace文件对应起来
- calculate_features_SI.ipynb